In [1]:
import pandas as pd
import numpy as np

# --- 1. DATA CLEANING & REFINEMENT ---
def clean_income_data(file_path):
    # Load the census dataset
    df = pd.read_csv('income-data.csv')
    
    # Remove duplicates to ensure unique behavioral observations
    df = df.drop_duplicates()
    
    # Standardize 'income' column: Remove trailing periods and whitespace
    # This is critical for binary classification accuracy
    df['income'] = df['income'].str.strip().str.replace('.', '', regex=False)
    
    # Handle missing values: Census '?' is replaced with 'Unknown'
    # We do not drop them because 'Unknown' work history is a data point in itself
    df = df.replace(['?', ' ?'], 'Unknown')
    
    # Normalize text: Strip whitespace from all categorical features
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].str.strip()
        
    return df

# --- 2. SELECTIVE FEATURE ENGINEERING ---
def engineer_money_drivers(df):
    # AGE GROUPS: Defining the life-cycle of money
    df['age_group'] = pd.cut(df['age'], bins=[0, 25, 45, 65, 100], 
                             labels=['Young', 'Adult', 'Senior', 'Elderly'])
    
    # WORK INTENSITY: Testing the limits of labor
    df['work_intensity'] = pd.cut(df['hours-per-week'], bins=[0, 34, 40, 60, 100], 
                                  labels=['Part-time', 'Full-time', 'Overtime', 'Extreme'])
    
    # EDUCATION TIER: Measuring intellectual leverage
    # High = Degree holders; Basic/HS = No degree
    edu_map = {
        'Bachelors': 'High', 'Masters': 'High', 'Doctorate': 'High', 'Prof-school': 'High',
        'Assoc-voc': 'Medium', 'Assoc-acdm': 'Medium', 'Some-college': 'Medium'
    }
    df['education_tier'] = df['education'].map(edu_map).fillna('Basic/HS')
    
    # IS HIGH INCOME: Target variable for Machine Learning (1 if >50K, else 0)
    df['is_high_income'] = (df['income'] == '>50K').astype(int)
    
    return df

In [2]:
import seaborn as sns
import matplotlib.pyplot as plt

def perform_eda(df):
    sns.set_theme(style="whitegrid")
    
    # VISUALIZING THE HUSTLE MYTH
    plt.figure(figsize=(10, 5))
    hustle_plot = df.groupby('work_intensity')['is_high_income'].mean() * 100
    sns.lineplot(x=hustle_plot.index, y=hustle_plot.values, marker='o', color='orange')
    plt.title("Income Probability vs. Work Hours")
    plt.ylabel("Chance of Earning >$50K (%)")
    plt.savefig('hustle_myth.png')
    
    # VISUALIZING THE EDUCATION PREMIUM
    plt.figure(figsize=(10, 5))
    edu_plot = df.groupby('education_tier')['is_high_income'].mean() * 100
    sns.barplot(x=edu_plot.index, y=edu_plot.values, palette='viridis')
    plt.title("Income Probability by Education Tier")
    plt.savefig('education_premium.png')

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

def train_money_predictor(df):
    # Prepare features: Drop non-behavioral columns and redundant features
    X = df.drop(['age', 'hours-per-week', 'education-num', 'is_high_income'], axis=1) # Example drop
    # Convert categories to numbers (One-Hot Encoding)
    X = pd.get_dummies(df[['age_group', 'work_intensity', 'education_tier', 'marital-status']])
    y = df['is_high_income']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Train Model with balanced weights to account for the minority of high earners
    model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    model.fit(X_train, y_train)
    
    return model